#**SENTIMENT ANALYSIS**
---
---
DL Assignment 4

###1. Import Libraries
---


In [6]:
# Basic Libraries
import numpy as np
import pandas as pd
import re
import string

# NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Sklearn Libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,classification_report,f1_score

# Download stopwords
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

###2. Loading the Dataset
---

In [7]:
df= pd.read_csv('/content/drive/MyDrive/datasets/nlp_dataset.csv')

In [8]:
df.head(5)

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5937 entries, 0 to 5936
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Comment  5937 non-null   object
 1   Emotion  5937 non-null   object
dtypes: object(2)
memory usage: 92.9+ KB


### 3. Data preprocessing
---
Preprocessing steps performed:

1. Lowercasing – Converts all text to lowercase to maintain uniformity.

2. Removing punctuation – Eliminates unnecessary symbols.

3. Removing numbers – Numbers usually don’t contribute to emotion classification.

4. Tokenization – Splitting text into individual words.

5. Stopword removal – Removes common words like "is", "the", "and" which do not carry strong sentiment meaning.

These steps reduce noise in text data and improve model performance by focusing only on meaningful words.

In [15]:
# Cleaning Function
#----------------------------

stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = re.sub(f"[{string.punctuation}]", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)


In [17]:
import nltk
nltk.download('punkt_tab')
# Apply cleaning
df['clean_text'] = df['Comment'].apply(clean_text)

df.head(10)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,Comment,Emotion,clean_text
0,i seriously hate one subject to death but now ...,fear,seriously hate one subject death feel reluctan...
1,im so full of life i feel appalled,anger,im full life feel appalled
2,i sit here to write i start to dig out my feel...,fear,sit write start dig feelings think afraid acce...
3,ive been really angry with r and i feel like a...,joy,ive really angry r feel like idiot trusting fi...
4,i feel suspicious if there is no one outside l...,fear,feel suspicious one outside like rapture happe...
5,i feel jealous becasue i wanted that kind of l...,anger,feel jealous becasue wanted kind love true con...
6,when a friend of mine keeps telling me morbid ...,anger,friend mine keeps telling morbid things happen...
7,i finally fell asleep feeling angry useless an...,anger,finally fell asleep feeling angry useless stil...
8,i feel a bit annoyed and antsy in a good way,anger,feel bit annoyed antsy good way
9,i feel like i ve regained another vital part o...,joy,feel like regained another vital part life living


###4. Feature Extraction
---
We use TF-IDF Vectorizer.

TF-IDF (Term Frequency - Inverse Document Frequency):

- Converts text into numerical vectors.

- Gives higher weight to important words.

- Reduces importance of frequently occurring words.

This helps the model focus on words that are more meaningful for emotion classification.

In [18]:
# Define features and target
x = df['clean_text']
y = df['Emotion']

# Apply TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
x_tfidf = vectorizer.fit_transform(x)

print("Feature shape:", x_tfidf.shape)

Feature shape: (5937, 5000)


###5. Train-Test Split
---

In [19]:
x_train,x_test,y_train,y_test = train_test_split(x_tfidf,
                                                 y,
                                                 test_size=0.2,
                                                 random_state=42)

###6. Model Development
---


####6.1 **Naive Bayes**
Naive Bayes is a probabilistic classifier based on Bayes' Theorem.
It works very well for text classification because it handles word frequencies efficiently.

In [22]:
nb_model = MultinomialNB()
nb_model.fit(x_train, y_train)

nb_pred = nb_model.predict(x_test)

####6.2 **Support Vector Machine**
SVM finds the optimal hyperplane that separates classes.
It performs well in high-dimensional spaces like text data.

In [23]:
svm_model = SVC(kernel='linear')
svm_model.fit(x_train, y_train)

svm_pred = svm_model.predict(x_test)

###7. Model Evaluation
---

####7.1 Naive Bayes Performance


In [25]:
print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))
print("Naive Bayes F1 Score:", f1_score(y_test, nb_pred, average='weighted'))
print("\nClassification Report:\n", classification_report(y_test, nb_pred))

Naive Bayes Accuracy: 0.9090909090909091
Naive Bayes F1 Score: 0.9089193266783082

Classification Report:
               precision    recall  f1-score   support

       anger       0.88      0.95      0.91       392
        fear       0.91      0.92      0.92       416
         joy       0.94      0.86      0.90       380

    accuracy                           0.91      1188
   macro avg       0.91      0.91      0.91      1188
weighted avg       0.91      0.91      0.91      1188



####7.2 SVM Performance

In [26]:

print("SVM Accuracy:", accuracy_score(y_test, svm_pred))
print("SVM F1 Score:", f1_score(y_test, svm_pred, average='weighted'))
print("\nClassification Report:\n", classification_report(y_test, svm_pred))

SVM Accuracy: 0.946969696969697
SVM F1 Score: 0.9469121249090242

Classification Report:
               precision    recall  f1-score   support

       anger       0.93      0.96      0.94       392
        fear       0.97      0.92      0.94       416
         joy       0.95      0.96      0.95       380

    accuracy                           0.95      1188
   macro avg       0.95      0.95      0.95      1188
weighted avg       0.95      0.95      0.95      1188



###8. Model Comparison
---

In [27]:
# Naive Bayes metrics
nb_accuracy = accuracy_score(y_test, nb_pred)
nb_f1 = f1_score(y_test, nb_pred, average='weighted')

# SVM metrics
svm_accuracy = accuracy_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred, average='weighted')

# Comparison Table
comparison_df = pd.DataFrame({
    "Model": ["Naive Bayes", "Support Vector Machine"],
    "Accuracy": [nb_accuracy, svm_accuracy],
    "F1-Score (Weighted)": [nb_f1, svm_f1]
})

comparison_df


,Model,Accuracy,F1-Score (Weighted)
0,Naive Bayes,0.909091,0.908919
1,Support Vector Machine,0.946970,0.946912


#### Comparison summary
- The SVM model achieved higher accuracy and F1-score compared to Naive Bayes.
- This indicates that SVM handles high-dimensional TF-IDF features more effectively.
- However, Naive Bayes is faster and computationally efficient.

### **Conclusion:**


####In this assignment, sentiment analysis was performed using TF-IDF feature extraction and two machine learning models: Naive Bayes and Support Vector Machine. After preprocessing and vectorization, both models were evaluated using accuracy and F1-score. Among the two, SVM performed better due to its ability to handle high-dimensional sparse text data effectively. This demonstrates that SVM is well-suited for emotion classification tasks.